[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gregwchase/dynatrace-pubsec-summit-2026/blob/main/notebooks/02_evals.ipynb)

# Lab 2: Measure Performance with Evals
**Traces to Insights: A Hands-On Workshop with Arize AX** · Dynatrace Public Sector Summit 2026 · ⏱️ ~35 min

**In this lab you will:**
- Pull your traces out of Arize into a DataFrame
- Write **code-based evaluators** (fast, free, deterministic)
- Build **LLM-as-a-judge** evaluators for **hallucination** and **tool-call accuracy**
- Log eval results back to Arize so they're attached to every trace
- **Promote them to online evals**: create evaluators and tasks in Arize with the SDK so every new trace is scored automatically

In [ ]:
# --- Setup: works in Google Colab or a local Jupyter environment ---
import os, sys, subprocess

if "google.colab" in sys.modules:
    REPO_DIR = "/content/dynatrace-pubsec-summit-2026"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/gregwchase/dynatrace-pubsec-summit-2026.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif not os.path.exists("civic_agent") and os.path.exists("../civic_agent"):
    os.chdir("..")  # running locally from notebooks/

sys.path.insert(0, os.getcwd())
from workshop_utils import load_credentials

# Model provider for the agent and the judges: "openai", "anthropic", "azure", or "bedrock".
# Leave blank to use LLM_PROVIDER from .env / Colab Secrets (default: openai).
LLM_PROVIDER = ""
PROJECT = load_credentials(provider=LLM_PROVIDER)  # prompts for keys if they aren't in .env / Colab Secrets

## 1. Export traces

We pull every span from the last 24 hours, then collapse each trace into one row: the resident's question, the final answer, which tools ran, and what those tools returned (`tool_context`).

In [ ]:
from workshop_utils import export_spans, root_spans

spans = export_spans(hours=24)
traces = root_spans(spans)
traces["tools_called_str"] = traces["tool_calls"].apply(lambda t: ", ".join(t) or "(none)")
traces[["input", "output", "tools_called_str", "guardrail_triggered"]].head(10)

## 2. Code-based evaluators

Use code when the rule is objective. These run in milliseconds and cost nothing, so you can run them on 100% of traffic.

1. **`cites_source`**: if tools returned a program record, does the answer include the official source link?
2. **`eligibility_tool_used`**: if the answer makes an eligibility claim ("you qualify", "you're eligible"), did the agent actually call `check_eligibility`? Or did it do the math itself?

In [ ]:
import re
from phoenix.evals import create_evaluator

@create_evaluator(name="cites_source", kind="code")
def cites_source(output: str, tool_context: str) -> float:
    returned_program = "source_url" in tool_context
    if not returned_program:
        return 1.0  # nothing to cite, so this is a pass
    return 1.0 if "dcs.columbia.example.gov" in output else 0.0

ELIGIBILITY_CLAIM = re.compile(r"\b(you|you're|you are|your household|you'd|we|they)\b.{0,40}\b(qualif|eligib)", re.I)

@create_evaluator(name="eligibility_tool_used", kind="code")
def eligibility_tool_used(output: str, tools_called_str: str) -> float:
    makes_claim = bool(ELIGIBILITY_CLAIM.search(output))
    if not makes_claim:
        return 1.0
    return 1.0 if "check_eligibility" in tools_called_str else 0.0

## 3. LLM-as-a-judge evaluators

Some qualities need judgment. We use a strong model as the judge (`CIVIC_JUDGE_MODEL`), give it a focused rubric, and ask it for a **label + explanation**. The explanation is what makes an eval useful for debugging.

### 3a. Hallucination: is the answer grounded in what the tools returned?

In [ ]:
from phoenix.evals import create_classifier
from workshop_utils import judge_model

# The judge model for your provider (e.g. gpt-4.1 or claude-opus-5-5). With OpenAI only, this is
# the same as: from phoenix.evals import LLM; judge = LLM(provider="openai", model="gpt-4.1")
judge = judge_model()

HALLUCINATION_TEMPLATE = """You are auditing a government benefits assistant. Its answers must be
based ONLY on information returned by its official tools, never on outside knowledge.

[Resident question]: {{input}}
[Tool results the assistant received]: {{tool_context}}
[Assistant answer]: {{output}}

Does the answer state any program, rule, amount, date, process, or recommendation that is
NOT supported by the tool results? General courtesy, a referral to the DCS hotline, or a
request for more information does not count as hallucination.

Respond "hallucinated" if the answer contains unsupported claims, otherwise "factual"."""

hallucination = create_classifier(
    name="hallucination",
    prompt_template=HALLUCINATION_TEMPLATE,
    llm=judge,
    choices={"factual": 1.0, "hallucinated": 0.0},
)

### 3b. Tool-call accuracy: did the agent pick the right tools for the question?

In [ ]:
TOOL_ACCURACY_TEMPLATE = """You are evaluating whether an AI agent used its tools correctly.

Available tools:
- search_programs(query): look up DCS program details. Required for ANY question about a program.
- check_eligibility(program_id, household_size, monthly_income): required for ANY eligibility
  determination. If household size or income is missing, the agent should ask instead.
- get_application_status(application_id): required for application status questions.
- find_office(zip_code): required for office location or hours questions.

[Resident question]: {{input}}
[Tools the agent called, in order]: {{tools_called_str}}
[Agent's final answer]: {{output}}

Respond "correct" if the agent called every tool the question required (and none that were
clearly wrong), or appropriately asked for missing information. Otherwise respond "incorrect"."""

tool_accuracy = create_classifier(
    name="tool_call_accuracy",
    prompt_template=TOOL_ACCURACY_TEMPLATE,
    llm=judge,
    choices={"correct": 1.0, "incorrect": 0.0},
)

## 4. Run all four evaluators

Guardrail-blocked traces never reached the LLM, so we skip them.

In [ ]:
from phoenix.evals import evaluate_dataframe

to_eval = traces[traces["guardrail_triggered"].fillna("none") == "none"].copy()
to_eval["output"] = to_eval["output"].fillna("")

results = evaluate_dataframe(
    dataframe=to_eval,
    evaluators=[cites_source, eligibility_tool_used, hallucination, tool_accuracy],
)

EVALS = ["cites_source", "eligibility_tool_used", "hallucination", "tool_call_accuracy"]
summary = {name: results[f"{name}_score"].apply(lambda s: s.get("score") if isinstance(s, dict) else None).mean() for name in EVALS}
print("Pass rate by evaluator (v1 prompt):")
for k, v in summary.items():
    print(f"  {k:<24} {v:.0%}")

Look at a few failures and read the judge's explanation:

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)

def failures(name, n=5):
    s = results[f"{name}_score"]
    mask = s.apply(lambda x: isinstance(x, dict) and x.get("score") == 0)
    out = results.loc[mask, ["input", "output"]].copy()
    out["explanation"] = s[mask].apply(lambda x: x.get("explanation"))
    return out.head(n)

failures("hallucination")

In [ ]:
failures("tool_call_accuracy")

## 5. Log evals back to Arize

Each eval is attached to the root span through `context.span_id`. In Arize you can then filter traces by eval label, chart pass rates on a dashboard, and add the failures to a dataset.

In [ ]:
from workshop_utils import to_arize_eval_columns, log_evals

eval_df = to_arize_eval_columns(results, EVALS)
log_evals(eval_df)
eval_df.head()

## 6. Explore in Arize 🔍

1. Open your project and refresh. You'll see new **eval columns** on the traces table.
2. Filter: `eval.hallucination.label = "hallucinated"`. These are your highest-risk traces.
3. Open one and read the explanation next to the trace.

## 7. Go online: evaluators and tasks in Arize

Everything so far ran **offline**, in this notebook, on traffic you already had. In production you want Arize to score **every new trace automatically**, without anyone running a notebook. That takes two resources, and you can create both with the SDK:

| Resource | What it is | SDK |
|---|---|---|
| **Evaluator** | A versioned judge definition stored in Arize: prompt template, labels, judge model | `client.evaluators.create_template_evaluator(...)` / `create_code_evaluator(...)` |
| **Task** | Runs evaluators on a project: which spans (filter), how often (continuous), and how many (sampling rate) | `client.tasks.create_evaluation_task(...)` |

Because both live in Arize, the whole team can see, version, and reuse them, and they keep running after this notebook closes.

### 7a. Connect a judge model

Online LLM judges run on Arize's side, so Arize needs stored credentials for the model provider: an **AI integration**. The helper creates one for your `LLM_PROVIDER` (OpenAI, Anthropic, Azure OpenAI, or Bedrock) with the key you already loaded, or reuses `ARIZE_AI_INTEGRATION_ID` if your facilitator gave you one.

> **Bedrock:** Arize can call Bedrock with a Bedrock API key or with an IAM role. If you're using normal AWS credentials, ask your facilitator for an integration ID.

In [ ]:
from workshop_utils import arize_client, participant_handle, online_judge_config

client = arize_client()
SPACE_ID = os.environ["ARIZE_SPACE_ID"]
HANDLE = participant_handle()

# AI integration + judge model + provider settings (e.g. the Azure deployment, the Bedrock region)
judge_llm = online_judge_config(client)
print("Online judge model:", judge_llm.model_name)

### 7b. Create the LLM-as-a-judge evaluators

We reuse the **exact rubrics** from section 3, so offline and online scores mean the same thing. Two differences:

- Arize templates use **single braces**, `{input}`, instead of `{{input}}`.
- The template names generic variables (`input`, `output`, `tool_context`). The **task** maps them to span attributes in step 7d, so the same evaluator can be reused on other projects.

Each evaluator also has a **result name**, the column its scores land in. Arize requires result names to be unique across the whole space, and everyone at the workshop shares one, so yours get your handle as a suffix: `eval.hallucination_<handle>.*`. Online and notebook scores then sit side by side on the same span, which makes them easy to compare. `root_spans()` merges them back into `eval.hallucination.*` for the later labs.

In [ ]:
from arize.evaluators.types import OptimizationDirection, TemplateConfigInput
from workshop_utils import find_evaluator, online_eval_name

def to_arize_template(t: str) -> str:
    return t.replace("{{", "{").replace("}}", "}")

def get_or_create_template_evaluator(name, eval_name, template, choices, description):
    existing = find_evaluator(client, name)
    if existing:
        print(f"Reusing evaluator '{name}'")
        return existing
    ev = client.evaluators.create_template_evaluator(
        name=name,
        space=SPACE_ID,
        description=description,
        commit_message="Promoted from Lab 2 notebook",
        template_config=TemplateConfigInput(
            name=online_eval_name(eval_name),   # result column: eval.<eval_name>_<handle>.label
            template=to_arize_template(template),
            classification_choices=choices,
            include_explanations=True,
            use_function_calling=True,
            direction=OptimizationDirection.MAXIMIZE,
            llm_config=judge_llm,
        ),
    )
    print(f"Created evaluator '{name}' ({ev.id})")
    return ev

hallucination_ev = get_or_create_template_evaluator(
    f"civicassist-{HANDLE}-hallucination", "hallucination", HALLUCINATION_TEMPLATE,
    {"factual": 1, "hallucinated": 0}, "Answer is grounded only in tool results",
)
tool_accuracy_ev = get_or_create_template_evaluator(
    f"civicassist-{HANDLE}-tool-call-accuracy", "tool_call_accuracy", TOOL_ACCURACY_TEMPLATE,
    {"correct": 1, "incorrect": 0}, "Agent called the tools the question required",
)

### 7c. Create a code evaluator

Arize has **managed code evaluators** (Matches Regex, Contains Any Keyword, Contains All Keywords, JSON Parseable, Exact Match) that run without any LLM calls. `cites_source` maps directly onto **Contains Any Keyword**. We'll use a filter in the task so it only checks answers where a tool returned a program record.

> `eligibility_tool_used` has conditional logic ("*if* the answer claims eligibility, *then* the tool must have been called"), so it needs a **custom code evaluator**: your own Python class running in Arize. Custom code evaluators are an Arize AX Enterprise feature, so this lab keeps that eval offline. The LLM `tool_call_accuracy` judge covers the same failure online.

In [ ]:
from arize.evaluators.types import CodeConfigRequest, ManagedCodeConfigRequest, ManagedCodeEvaluator, StaticParamRequest

CITES_NAME = f"civicassist-{HANDLE}-cites-source"
cites_source_ev = find_evaluator(client, CITES_NAME)
if cites_source_ev:
    print(f"Reusing evaluator '{CITES_NAME}'")
else:
    cites_source_ev = client.evaluators.create_code_evaluator(
        name=CITES_NAME,
        space=SPACE_ID,
        description="Answer includes the official DCS source link",
        commit_message="Promoted from Lab 2 notebook",
        # Wrap in CodeConfigRequest explicitly: passing the inner config on its own
        # serializes to null in arize 8.62.
        code_config=CodeConfigRequest(ManagedCodeConfigRequest(
            type="MANAGED",
            name=online_eval_name("cites_source"),   # result column, unique in the space
            managed_evaluator=ManagedCodeEvaluator.CONTAINS_ANY_KEYWORD,
            variables=["attributes.output.value"],
            static_params=[StaticParamRequest.from_dict(
                {"name": "keywords", "type": "STRING_ARRAY", "default_value": ["dcs.columbia.example.gov"]}
            )],
        )),
    )
    print(f"Created evaluator '{CITES_NAME}' ({cites_source_ev.id})")

### 7d. Create the online tasks

A task decides **which spans** get scored and **how**:

- **`query_filter`**: score only the root `civicassist.handle_request` span (one per request), and skip guardrail-blocked requests.
- **`column_mappings`**: wire each template variable to a span attribute. The agent writes the tool results into `metadata.tool_context` on the root span, so a single span has everything the judge needs.
- **`is_continuous=True`**: score new traces as they arrive.
- **`sampling_rate`**: 100% for the workshop. In production you'd often sample LLM judges (say 10–20%) to control cost, and run cheap code evals on everything.

LLM-judge and code evaluators run as separate task types, so we create two tasks.

In [ ]:
from arize.tasks.types import TaskEvaluatorInput, TaskType
from workshop_utils import find_task

ROOT_SPANS = "name = 'civicassist.handle_request' AND attributes.metadata.guardrail_triggered = 'none'"

COLUMN_MAPPINGS = {
    "input": "attributes.input.value",
    "output": "attributes.output.value",
    "tool_context": "attributes.metadata.tool_context",
    "tools_called_str": "attributes.metadata.tools_called",
}

def get_or_create_task(name, task_type, evaluators, sampling_rate):
    existing = find_task(client, name)
    if existing:
        print(f"Reusing task '{name}'")
        return existing
    task = client.tasks.create_evaluation_task(
        name=name,
        task_type=task_type,
        project=PROJECT,
        space=SPACE_ID,
        evaluators=evaluators,
        query_filter=ROOT_SPANS,
        is_continuous=True,
        sampling_rate=sampling_rate,
    )
    print(f"Created task '{name}' ({task.id})")
    return task

judge_task = get_or_create_task(
    f"civicassist-{HANDLE}-online-judges",
    TaskType.TEMPLATE_EVALUATION,
    [
        TaskEvaluatorInput.from_dict({"evaluator_id": hallucination_ev.id, "column_mappings": COLUMN_MAPPINGS}),
        TaskEvaluatorInput.from_dict({"evaluator_id": tool_accuracy_ev.id, "column_mappings": COLUMN_MAPPINGS}),
    ],
    sampling_rate=1.0,
)

code_task = get_or_create_task(
    f"civicassist-{HANDLE}-online-code-evals",
    TaskType.CODE_EVALUATION,
    [
        TaskEvaluatorInput.from_dict({
            "evaluator_id": cites_source_ev.id,
            # per-evaluator filter, AND-ed with the task filter
            "query_filter": "attributes.metadata.returned_program_record = 'true'",
        }),
    ],
    sampling_rate=1.0,
)

### 7e. Backfill: score the traffic you already have

Continuous tasks only score **new** traces. To check that the online setup works and agrees with the notebook, trigger a one-time run over the last 24 hours. The online scores land in your `_<handle>` columns, next to the notebook's scores on the same spans, so you can compare them. `override_evaluations=True` re-scores spans that an earlier run already scored. This takes 1–3 minutes.

In [ ]:
from datetime import datetime, timedelta, timezone

def backfill(task, hours=24):
    end = datetime.now(timezone.utc)
    run = client.tasks.trigger_run(
        task=task.id,
        space=SPACE_ID,
        data_start_time=end - timedelta(hours=hours),
        data_end_time=end,
        override_evaluations=True,
    )
    run = client.tasks.wait_for_run(run_id=run.id, poll_interval=10, timeout=600)
    print(f"{task.name}: {run.status}  successes={run.num_successes} errors={run.num_errors} "
          f"skipped={run.num_skipped} {run.failure_reason or ''}")
    return run

judge_run = backfill(judge_task)
code_run = backfill(code_task)

| If you see | It means |
|---|---|
| `COMPLETED`, successes > 0 | 🎉 Scores are on your traces. Refresh the project. |
| `COMPLETED`, 0 successes, many skipped | The filter matched spans, but a column mapping didn't resolve. Open a root span and check the attribute paths. |
| `COMPLETED`, 0 spans | Nothing matched the filter or time window yet. Wait a minute for ingestion and re-run. |
| `CANCELLED` within seconds | The AI integration's credentials are invalid. |

### 7f. Watch it work on live traffic

Send a few new requests. **No eval code runs in this notebook.** Arize scores them on its own.

In [ ]:
from civic_agent import setup_tracing, ask, flush

setup_tracing(PROJECT)
live = [ask(q, user_id="online-eval-demo") for q in [
    "Does DCS offer a first-time homebuyer program?",
    "Family of 3 making $3,000 a month. Can we get food assistance?",
    "What documents do I need for HEAP?",
]]
flush()
LIVE_TRACE_IDS = [r["trace_id"] for r in live]
print("Sent. Give Arize 2–5 minutes, then run the next cell.")

In [ ]:
# Re-run this cell until the eval columns fill in.
from workshop_utils import export_spans, root_spans

latest = root_spans(export_spans(hours=1))
latest = latest[latest["context.trace_id"].isin(LIVE_TRACE_IDS)]
cols = ["input", "output"] + [c for c in latest.columns if c.startswith("eval.") and c.endswith(".label")]
latest[cols]

Open **Evals & Tasks** in Arize to see your evaluators (with version history) and tasks (with run history). From here on, your online tasks keep scoring every new trace. In Lab 4, the v2 traffic is scored automatically.

> 🧹 **After the workshop:** continuous tasks keep calling the judge model on every new trace. Lab 5 ends with a cleanup cell that deletes your tasks and evaluators.

🤔 **Discuss:** which evals would you run on 100% of traffic, and which would you sample? What would you alert on?

**Next:** [Lab 3: Iterate with datasets and experiments →](03_datasets_experiments.ipynb)